# 00 — Configuração do ambiente

**Squad 2 — Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Tabela:** `ecommerce_itens_pedido`

Módulo de configuração. Os demais notebooks carregam este via `%run`.

**Acesso ao Data Lake.** A leitura é feita pelo próprio Spark (`spark.read`), distribuída entre os
executores. As credenciais OAuth do service principal vão num dicionário, `adls_options`, passado com
`.options(**adls_options)` em cada leitura e escrita. `spark.conf.set()` não é usado porque o
Databricks Serverless bloqueia as chaves `fs.azure.account.*` na sessão.

| # | Garantia | Como |
|---|---|---|
| C1 | Nenhuma credencial em código ou em saída de célula | `.env` lido para o objeto `CONFIG`, sem passar por `os.environ` |
| C2 | Acesso ao lake sem `spark.conf.set` | credenciais por chamada em `adls_options` |
| C3 | Tabela, chave, schema e caminhos num lugar só | bloco de parâmetros; `01` a `06` derivam dele |
| C4 | Conexão com o SQL Server pelo conector nativo | `opcoes_sqlserver(tabela)` |

## Parâmetros

Tudo o que identifica a tabela fica aqui. `SCHEMA_ORIGEM` é o schema do parquet do gerador: a Bronze
lê sempre com ele, para que o tipo das colunas não mude de um lote para outro. `ecommerce_itens_pedido`
não tem coluna de data, então não passa pelo problema de `TIMESTAMP(NANOS)` das outras tabelas.

As camadas ficam em pastas Delta no container da squad (`squad2/grupo2/...`). A Silver de itens depende
da Silver de pedidos (regra T2) e da Silver de produtos (regra T4), que ficam na raiz do container
(`squad2/silver/...`); os caminhos delas ficam aqui também.

In [ ]:
# ==========================================================
#  PARAMETROS - o unico bloco a alterar para trocar de tabela
# ==========================================================
SQUAD, GRUPO = "squad2", "grupo2"
TABELA, CHAVE_PRIMARIA = "ecommerce_itens_pedido", "id_item_pedido"
SCHEMA_ORIGEM = "id_item_pedido bigint, id_pedido bigint, sku string, quantidade bigint, preco_unitario double, desconto_aplicado double"
FUSO_NEGOCIO = "America/Sao_Paulo"

# Origem: o gerador grava um lote a cada ~120s em raw/real-time-data/AAAA/MM/DD/HHMMSS/<tabela>.parquet
CONTAINER, PREFIXO, PADRAO_LOTES = "raw", "real-time-data", "*/*/*/*"   # um nivel de glob por componente do carimbo

# Destinos
CATALOGO = "workspace"                                       # Bronze: tabela gerenciada do Unity Catalog
TABELA_BRONZE = f"{CATALOGO}.{SQUAD}.bronze_{TABELA}"
TABELA_DESTINO, MODO_ESCRITA = f"{SQUAD}.{TABELA}", "overwrite"   # carga direta do 03
# ==========================================================

print(f"Origem : {CONTAINER}/{PREFIXO}/{PADRAO_LOTES}/{TABELA}.parquet  (chave {CHAVE_PRIMARIA})")
print(f"Bronze : {TABELA_BRONZE}\nCamadas: {SQUAD}/{GRUPO}/{{silver,quarantine,waiting,gold}}/\nSQL    : {TABELA_DESTINO}")

## Busca do `.env`

O caminho do `.env` muda conforme quem clonou o repositório no Workspace. A busca sobe a partir da
pasta do notebook e depois tenta `/Workspace/Repos/*/<repo>/.env`. O arquivo é lido por um parser simples
(`CHAVE=valor`, aspas opcionais, `#` para comentário), sem instalar pacote: um `%pip` com reinício do Python
derruba o kernel quando este notebook é chamado por `%run`. As credenciais ficam em `CONFIG` e nunca
aparecem em saída.

In [ ]:
import os
from pathlib import Path

REPO = "estagio-empregadados-turma-2"


def descobrir_env():
    candidatos = []
    try:
        aqui = Path(os.getcwd()).resolve()
        candidatos += [p / ".env" for p in [aqui, *aqui.parents]]
    except Exception:
        pass
    if Path("/Workspace/Repos").is_dir():
        candidatos += sorted(Path("/Workspace/Repos").glob(f"*/{REPO}/.env"))
    candidatos.append(Path("/Workspace/Shared") / REPO / ".env")
    return next((c for c in candidatos if c.is_file()), None)


caminho_env = descobrir_env()
if caminho_env is None:
    raise FileNotFoundError("Nenhum .env encontrado. Copie o .env.example da raiz do repositorio para .env e preencha os valores.")


def ler_env(caminho):
    config = {}
    for linha in caminho.read_text(encoding="utf-8-sig").splitlines():
        linha = linha.strip()
        if not linha or linha.startswith("#") or "=" not in linha:
            continue
        chave, valor = linha.removeprefix("export ").split("=", 1)
        valor = valor.strip()
        if len(valor) > 1 and valor[0] == valor[-1] and valor[0] in ("'", '"'):
            valor = valor[1:-1]
        else:
            valor = valor.split(" #", 1)[0].strip()
        if valor:
            config[chave.strip()] = valor
    return config


CONFIG = ler_env(caminho_env)


def exigir(chaves, contexto):
    faltando = [c for c in chaves if not CONFIG.get(c)]
    if faltando:
        raise ValueError(f"{contexto}: variaveis ausentes no .env -> {', '.join(faltando)}")


OBRIGATORIAS_SQL = ["JDBC_HOSTNAME", "JDBC_DATABASE", "JDBC_USERNAME", "JDBC_PASSWORD"]
exigir(["ADLS_STORAGE_ACCOUNT", "ADLS_CLIENT_ID", "ADLS_TENANT_ID", "ADLS_CLIENT_SECRET"], "ADLS Gen2")
print(f"[ok] .env carregado de {caminho_env}; credenciais do ADLS presentes")
try:
    exigir(OBRIGATORIAS_SQL, "Azure SQL Server"); print("[ok] credenciais do SQL Server presentes")
except ValueError as e:
    print(f"[aviso] {e}")  # so o 03 e o 06 precisam do SQL Server

## Credenciais por chamada e caminhos

`adls_options` são as cinco chaves do `ClientCredsTokenProvider` do conector ABFS. O nome da storage
account vem do `.env`, nunca de literal no código: o repositório é público.

`ler_delta` devolve `None` quando a pasta ainda não existe (primeira execução), e deixa passar erros de
acesso. O `df.schema` dentro do `try` força a análise ali mesmo: no Serverless a leitura é preguiçosa.

In [ ]:
_sufixo = f"{CONFIG['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{_sufixo}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{_sufixo}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{_sufixo}": CONFIG["ADLS_CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{_sufixo}": CONFIG["ADLS_CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{_sufixo}": f"https://login.microsoftonline.com/{CONFIG['ADLS_TENANT_ID']}/oauth2/token",
}

CAMINHO_ORIGEM = f"abfss://{CONTAINER}@{_sufixo}/{PREFIXO}/{PADRAO_LOTES}/{TABELA}.parquet"
BASE = f"abfss://{SQUAD}@{_sufixo}/{GRUPO}"
CAMINHO_SILVER, CAMINHO_QUARENTENA, CAMINHO_ESPERA = (f"{BASE}/{camada}/{TABELA}" for camada in ("silver", "quarantine", "waiting"))
# Silvers de pedidos e produtos mantidas na raiz do container da squad (fora de grupo2/)
RAIZ_SQUAD = f"abfss://{SQUAD}@{_sufixo}"
CAMINHO_SILVER_PEDIDOS, CAMINHO_SILVER_PRODUTOS = f"{RAIZ_SQUAD}/silver/ecommerce_pedidos", f"{RAIZ_SQUAD}/silver/ecommerce_produtos"
CAMINHO_QUARENTENA_PEDIDOS = f"{RAIZ_SQUAD}/quarantine/ecommerce_pedidos"


def ler_delta(caminho):
    try:
        df = spark.read.format("delta").options(**adls_options).load(caminho)
        df.schema
        return df
    except Exception as e:
        if any(t in str(e) for t in ("PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST", "DELTA_MISSING_DELTA_TABLE", "is not a Delta table", "does not exist")):
            return None
        raise


def opcoes_sqlserver(dbtable):
    exigir(OBRIGATORIAS_SQL, "Azure SQL Server")
    return {"host": CONFIG["JDBC_HOSTNAME"], "port": CONFIG.get("JDBC_PORT", "1433"), "database": CONFIG["JDBC_DATABASE"],
            "user": CONFIG["JDBC_USERNAME"], "password": CONFIG["JDBC_PASSWORD"], "dbtable": dbtable}


print("[ok] setup concluido: adls_options, caminhos, ler_delta() e opcoes_sqlserver() disponiveis")